In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path("..").resolve()))

from src.features import load_m3
from src.baseline import make_model, GROUP_CV, log_run
from sklearn.model_selection import cross_val_predict, cross_validate

df, docs, y, groups = load_m3("../data/raw/evtx_data.csv")
LABELS = sorted(y.unique())
oof = cross_val_predict(make_model(), docs, y, cv=GROUP_CV, groups=groups)

In [4]:
# Bài 10.1 — Các cặp nhầm lẫn lớn nhất
# Đề: tạo ma trận nhầm lẫn dạng bảng, xóa đường chéo, in 5 cặp nhầm lớn nhất. In tổng số event bị đoán là CA và số event thật là CA.
from sklearn.metrics import confusion_matrix

cm = pd.DataFrame(confusion_matrix(y, oof, labels=LABELS), index=LABELS, columns=LABELS)
arr = cm.to_numpy().copy()                               # copy để được phép sửa
np.fill_diagonal(arr, 0)                                 # bỏ đường chéo (đoán đúng)
pairs = pd.DataFrame(arr, index=LABELS, columns=LABELS).stack()   # (thật, đoán) -> số event
print(pairs.sort_values(ascending=False).head(5))
print("đoán thành Credential Access:", cm["Credential Access"].sum(),
      "| thật là Credential Access:", cm.loc["Credential Access"].sum())

Lateral Movement      Credential Access    643
Execution             Credential Access    357
Command and Control   Credential Access    258
Privilege Escalation  Defense Evasion      196
Defense Evasion       Credential Access    137
dtype: int64
đoán thành Credential Access: 1680 | thật là Credential Access: 283


In [ ]:
# Bài 10.2 — Top token của từng tactic
# Đề: fit mô hình trên toàn bộ dữ liệu, in 5 token có hệ số lớn nhất của mỗi tactic.
model_full = make_model(C=1.0).fit(docs, y)                # fit 1 lần, CHỈ để diễn giải

# lấy bước TF-IDF trong Pipeline theo tên đã đặt trong make_model.
# get_feature_names_out(): danh sách 1.278 token theo thứ tự cột. Giống Buổi 4.
vocab = model_full.named_steps["tfidf"].get_feature_names_out() 
clf = model_full.named_steps["clf"] # lấy bước Logistic Regression.

# enumerate : cho cả số thứ tự i lẫn tên tactic.
for i, tactic in enumerate(clf.classes_): # 8 tactic theo đúng thứ tự các hàng trong coef_.
    top = np.argsort(clf.coef_[i])[::-1][:5]               # 5 hệ số lớn nhất của lớp i
    print(f"{tactic[:20]:20s} | {', '.join(vocab[top])}")

Command and Control  | ch_remoteconnectionmanager, eid_3, harddiskvolume1, eid_5156, eid_4688
Credential Access    | cscript.exe, lsass.exe, appcmd.exe, eid_4768, eid_18456
Defense Evasion      | windowscoredeviceinfo.dll, eid_4663, eid_8, 0x00000001, eid_4826
Discovery            | 0x00001410, harddiskvolume2, eid_4661, 0x0000002d, eid_10
Execution            | eid_1040, eid_1042, ch_application, mshta.exe, rundll32.exe
Lateral Movement     | eid_5145, 0x00100081, ch_rdpcorets, bloodhound-win32-x64, eid_33205
Persistence          | eid_1, services.exe, eid_5136, osk.exe, eid_4662
Privilege Escalation | consent.exe, eid_7, uacme.exe, c, privesc


In [ ]:
# Kiểm tra nghi vấn harddiskvolume
vol = docs.str.contains(r"harddiskvolume\d", regex=True)
print("event có harddiskvolume:", vol.sum())
volume = docs.str.extract(r"(harddiskvolume\d)")[0].rename("volume")
print(pd.crosstab(volume, y).loc[:, lambda t: t.sum() > 0])   # chỉ giữ cột có số > 0

# Mỗi số volume gần như gắn với đúng một tactic: volume1 → C2, volume2 → Discovery, volume4 → Execution.
# Số volume phụ thuộc vào máy lab, mỗi lab lại chạy vài loại tactic. Xác nhận đây là đường tắt thật.

event có harddiskvolume: 93
EVTX_Tactic      Command and Control  Defense Evasion  Discovery  Execution
volume                                                                     
harddiskvolume1                   63                1          0          0
harddiskvolume2                    0                0         25          0
harddiskvolume4                    0                0          0          4


In [7]:
# Sửa đường tắt, chạy lại và ghi lại
docs_novol = docs.str.replace(r"\\device\\harddiskvolume\d+\\", r"<vol>\\", regex=True)
print(docs_novol[vol].iloc[0][:80])
assert not docs_novol.str.contains(r"harddiskvolume\d").any()   # shadowcopy vẫn giữ

s_new = cross_validate(make_model(), docs_novol, y, cv=GROUP_CV, groups=groups,
                       scoring="f1_macro")["test_score"]
s_old = cross_validate(make_model(), docs, y, cv=GROUP_CV, groups=groups,
                       scoring="f1_macro")["test_score"]
print(f"cũ : {s_old.mean():.3f} +/- {s_old.std():.3f}")
print(f"mới: {s_new.mean():.3f} +/- {s_new.std():.3f}")
log_run("A-novol", "StratifiedGroupKFold(5,rs=42)", s_new,
        C=1.0, note="harddiskvolume->vol")

oof_new = cross_val_predict(make_model(), docs_novol, y, cv=GROUP_CV, groups=groups)
from sklearn.metrics import f1_score
cmp = pd.DataFrame({"cũ": f1_score(y, oof, labels=LABELS, average=None),
                    "mới": f1_score(y, oof_new, labels=LABELS, average=None)},
                   index=LABELS).round(3)
print(cmp.to_string())

ch_security eid_5156 <vol>\windows\system32\svchost.exe
cũ : 0.206 +/- 0.033
mới: 0.202 +/- 0.034
                         cũ    mới
Command and Control   0.019  0.019
Credential Access     0.152  0.140
Defense Evasion       0.027  0.026
Discovery             0.071  0.063
Execution             0.059  0.059
Lateral Movement      0.441  0.441
Persistence           0.333  0.331
Privilege Escalation  0.615  0.586


In [8]:
# Top token sau khi sửa
m_new = make_model().fit(docs_novol, y)
v_new = m_new.named_steps["tfidf"].get_feature_names_out()
for tactic in ["Command and Control", "Discovery"]:
    i = list(m_new.named_steps["clf"].classes_).index(tactic)
    top = np.argsort(m_new.named_steps["clf"].coef_[i])[::-1][:5]
    print(f"{tactic[:20]:20s} | {', '.join(v_new[top])}")

Command and Control  | ch_remoteconnectionmanager, eid_3, eid_5156, eid_4688, eid_5
Discovery            | 0x00001410, <vol>, eid_4661, 0x0000002d, eid_10
